In [4]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import glob

In [ ]:



# Load parquet files
benign_df = pd.read_csv("processedDarabenign.csv")
mal_df = pd.read_csv("malware.csv")

print("Benign shape:", benign_df.shape)
print("Malware shape:", mal_df.shape)
 
# IMPORTANT: reset column names to be identical
benign_df.columns = range(benign_df.shape[1])
mal_df.columns = range(mal_df.shape[1])


# Concatenate safely
df = pd.concat([benign_df, mal_df], axis=0, ignore_index=True)

print("Final dataset shape:", df.shape)
#print(df["label"].value_counts())

print(df.shape)


#colomn 9505 is ur lable

ValueError: Invalid file path or buffer object type: <class 'list'>

In [ ]:

LABEL_COL = 9505

X = df.drop(columns=[LABEL_COL, 1])  # also drop hash column (1)
y = df[LABEL_COL]

# Compute correlation of each feature with label
correlations = X.corrwith(y)

# Sort by absolute correlation
corr_sorted = correlations.abs().sort_values(ascending=False)

print(corr_sorted.head(200))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X_reduced, y, test_size=0.2,random_state=42
)

model = RandomForestClassifier(
    n_estimators=200, n_jobs=-1, random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", model.score(X_test, y_test))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Report:\n", classification_report(y_test, y_pred))


In [ ]:
import numpy as np

y_train_shuffled = np.random.permutation(y_train)
model.fit(X_train, y_train_shuffled)
print("Accuracy with shuffled labels:", model.score(X_test, y_test))

In [ ]:

import shap
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# sample only 200 rows
X_sample = X_train.sample(10000)

# train a smaller RF just for SHAP
rf_shap = RandomForestClassifier(
    n_estimators=50,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf_shap.fit(X_train, y_train)

explainer = shap.TreeExplainer(rf_shap)


explainer = shap.TreeExplainer(rf_shap)

shap_values = explainer(X_sample)  # only malware class
malicious_shap_values = shap_values [:, :, 1]

shap.summary_plot(malicious_shap_values, X_sample)


In [ ]:
shap.summary_plot(shap_values[:, :, 1], X_sample, plot_type="bar")

In [ ]:
sample = X_test.iloc[0:10]
pred = model.predict(sample)
# 2. Iterate through the predictions

explainer = shap.TreeExplainer(model)
shap_explanations = explainer(sample, check_additivity=False)

for i, app_pred in enumerate(pred):
    status = "Malware" if app_pred == 1 else "Benign"
    
    # Get the SHAP values for this specific app (class 1: Malicious)
    # We look for the highest positive contributions for Malware
    # and highest negative contributions for Benign
    current_shap_values = shap_explanations.values[i, :, 1]
    feature_names = sample.columns
    
    # Sort features by their impact (absolute SHAP value)
    top_indices = np.argsort(np.abs(current_shap_values))[-3:][::-1]
    
    reasons = []
    for idx in top_indices:
        val = current_shap_values[idx]
        feature_name = feature_names[idx]
        direction = "contributed to Malware risk" if val > 0 else "suggested Benign behavior"
        reasons.append(f"{feature_name} ({direction} risk by {abs(val):.4f})%")
        

    print(f"App {i}: {status} detected.")
    print(f"   Reasoning: {', '.join(reasons)}\n")